# Baseline Preprocessing + Pruning Notebook

This notebook contains the preprocessing and pruning logic used by the baseline model, with comments.

It covers:
- missing-value imputation
- categorical encoding
- fixed feature pruning (remove `CGPA`, `Age`, `Semester_Credit_Load`)
- scaling and train/test prep

In [ ]:
# 1) Imports and paths
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = '/Users/edgarvidriales/Desktop/AuraCheck/auracheck'
DATA_PATH = '/Users/edgarvidriales/Desktop/AuraCheck/auracheck/Dataset/students_mental_health_survey_with_burnout_final.csv'

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)

In [ ]:
# 2) Define full and pruned feature sets
# Full candidate set before pruning
FEATURES_ALL = [
    'Course', 'Age', 'Gender', 'CGPA', 'Sleep_Quality', 'Physical_Activity',
    'Diet_Quality', 'Social_Support', 'Relationship_Status', 'Substance_Use',
    'Counseling_Service_Use', 'Family_History', 'Chronic_Illness',
    'Financial_Stress', 'Extracurricular_Involvement', 'Semester_Credit_Load',
    'Residence_Type'
]

# Pruned baseline set (VIF-informed removal)
REMOVED_BY_PRUNING = ['CGPA', 'Age', 'Semester_Credit_Load']
FEATURES_PRUNED = [f for f in FEATURES_ALL if f not in REMOVED_BY_PRUNING]

print('Removed:', REMOVED_BY_PRUNING)
print('Pruned feature count:', len(FEATURES_PRUNED))
print(FEATURES_PRUNED)

In [ ]:
# 3) Categorical encoding map used in baseline
ENCODING_MAP = {
    'Gender': {'Female': 0, 'Male': 1},
    'Sleep_Quality': {'Poor': 0, 'Average': 1, 'Good': 2},
    'Physical_Activity': {'Low': 0, 'Moderate': 1, 'High': 2},
    'Diet_Quality': {'Good': 0, 'Average': 1, 'Poor': 2},
    'Social_Support': {'High': 0, 'Moderate': 1, 'Low': 2},
    'Substance_Use': {'Never': 0, 'Unknown': 1, 'Occasionally': 2, 'Frequently': 3},
    'Counseling_Service_Use': {'Never': 0, 'Occasionally': 1, 'Frequently': 2},
    'Family_History': {'No': 0, 'Yes': 1},
    'Chronic_Illness': {'No': 0, 'Yes': 1},
    'Extracurricular_Involvement': {'High': 0, 'Moderate': 1, 'Low': 2},
    'Course': {'Business': 0, 'Computer Science': 1, 'Engineering': 2, 'Law': 3, 'Medical': 4, 'Others': 5},
    'Relationship_Status': {'In a Relationship': 0, 'Married': 1, 'Single': 2},
    'Residence_Type': {'Off-Campus': 0, 'On-Campus': 1, 'With Family': 2},
}

In [ ]:
# 4) Preprocessing function (imputation + encoding)
def preprocess(df_in: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """
    Preprocess baseline features:
    1) select columns
    2) impute missing values
       - categorical/object: 'Unknown'
       - numeric: median
    3) apply fixed categorical encoding map
    4) cast to float for model/scaler
    """
    X = df_in[features].copy()

    # Missing-value imputation
    for c in X.columns:
        if X[c].isnull().any():
            if X[c].dtype == object:
                X[c] = X[c].fillna('Unknown')
            else:
                X[c] = X[c].fillna(X[c].median())

    # Deterministic encoding; unknown categories fallback to 1
    for c, mp in ENCODING_MAP.items():
        if c in X.columns:
            X[c] = X[c].astype(str).map(mp).fillna(1)

    return X.astype(float)

In [ ]:
# 5) Build target and run preprocessing on pruned features
# Target: quartile-based 4-class label from burnout_raw_score
y = pd.qcut(df['burnout_raw_score'].astype(float), q=4, labels=[0, 1, 2, 3], duplicates='drop').astype(int)

# Use pruned features for baseline model
X_pruned = preprocess(df, FEATURES_PRUNED)

print('X_pruned shape:', X_pruned.shape)
print('y distribution:', y.value_counts().sort_index().to_dict())
X_pruned.head()

In [ ]:
# 6) Train/test split + scaling (pre-model pipeline)
X_train, X_test, y_train, y_test = train_test_split(
    X_pruned, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Scaled train mean (first 5 cols):', np.round(X_train_s.mean(axis=0)[:5], 6))

## Notes
- This notebook captures the **pruning + preprocessing** portion only.
- Full training/artifact export remains in `production_pruned_multinomial_baseline.py` (and its notebook variant).